# Conditional GAN (cGAN) — Summary
 - Dataset: `MNIST` normalized to [-1, 1]; labels retained for conditioning.
 - Generator: takes 100-d noise + label embedding (label → Embedding → Dense → Reshape), concatenates them, then uses `Conv2DTranspose` layers to produce 28×28 images.
 - Discriminator: takes image + label embedding (reshaped to image spatial dims), concatenates along channels, then classifies real vs. fake via Conv layers and a final dense logit.
 - Training: adversarial training with Binary Crossentropy loss and separate optimizers; `train_step` computes generator and discriminator losses and applies gradients. Noise should be generated per-batch (see notes).
 - Interactive: the notebook includes widget buttons that generate a digit by passing conditioned `label` + `noise` to `generator` and displaying the result.
 
 Notes / Gotchas:
 - When calling `generator([noise, label])`, ensure both inputs have the same batch dimension and are the same type (both tensors or both numpy arrays).
 - For interactive calls prefer a tensor label with explicit batch dim and integer dtype, e.g.: l`abel = tf.constant([[digit]], dtype=tf.int32)` (shape `(1,1)`), or use `np.array([[digit]], dtype=np.int32)` for NumPy inputs.
 - In the training loop, avoid fixed BATCH_SIZE noise if the last batch may be smaller; instead do `noise = tf.random.normal([tf.shape(labels)[0], 100])` or create the dataset with `drop_remainder=True` to guarantee uniform batch sizes."

# 1. Setup and Data Preparation
We normalize the MNIST images and prepare the labels for the conditional training.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

# Load MNIST
(train_images, train_labels), (_, _) = tf.keras.datasets.mnist.load_data()
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5 # Normalize to [-1, 1]

num_classes = 10
BATCH_SIZE = 256

# Create dataset and include labels
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.shuffle(60000).batch(BATCH_SIZE)


# 2. Conditional Architecture
In a cGAN, we use an Embedding layer to turn the class integer (0-9) into a vector, which is then concatenated with the noise (for the Generator) or the image features (for the Discriminator).


In [ ]:
def make_generator_model():
    # Label input
    label = layers.Input(shape=(1,))
    label_embedding = layers.Embedding(num_classes, 50)(label)
    label_embedding = layers.Dense(7*7*1)(label_embedding)
    label_embedding = layers.Reshape((7, 7, 1))(label_embedding)

    # Noise input
    noise = layers.Input(shape=(100,))
    gen = layers.Dense(7*7*256, use_bias=False)(noise)
    gen = layers.BatchNormalization()(gen)
    gen = layers.LeakyReLU()(gen)
    gen = layers.Reshape((7, 7, 256))(gen)

    # Merge label and noise
    concat = layers.Concatenate()([gen, label_embedding])

    x = layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False)(concat)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    out = layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh')(x)

    return tf.keras.Model([noise, label], out)

def make_discriminator_model():
    img = layers.Input(shape=(28, 28, 1))
    label = layers.Input(shape=(1,))

    label_embedding = layers.Embedding(num_classes, 50)(label)
    label_embedding = layers.Dense(28*28*1)(label_embedding)
    label_embedding = layers.Reshape((28, 28, 1))(label_embedding)

    # Concat image and label
    concat = layers.Concatenate()([img, label_embedding])

    x = layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same')(concat)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    out = layers.Dense(1)(x)

    return tf.keras.Model([img, label], out)

generator = make_generator_model()
discriminator = make_discriminator_model()

# 3. Training Logic
The loss functions remain the same as a standard GAN, but we pass both the image and its corresponding label during the forward pass.


In [ ]:
from tqdm import tqdm
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
gen_optimizer = tf.keras.optimizers.Adam(1e-4)
disc_optimizer = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(real_images, labels):
    batch_size = tf.shape(labels)[0]
    noise = tf.random.normal([batch_size, 100])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([???, ???], training=True). # What to send to the generator? 

        real_output = discriminator([???, ???], training=True) # What to send to the discriminator during real inputs? 
        fake_output = discriminator([???, ???], training=True) # What to send to the discriminator during fake inputs? 

        gen_loss = binary_cross_entropy(tf.ones_like(fake_output), fake_output)

        real_loss = binary_cross_entropy(tf.ones_like(real_output), real_output)
        fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
        disc_loss = real_loss + fake_loss

    generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))
    return gen_loss, disc_loss
# Training loop

NUM_EPOCHS = 20 # Increase this for better results (20-50 epochs recommended)

for epoch in range(NUM_EPOCHS): # Increased epochs lead to better quality
    pbar = tqdm(train_dataset, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for image_batch, label_batch in pbar:
        g_loss, d_loss = train_step(image_batch, label_batch)
        pbar.set_postfix({'g_loss': g_loss.numpy(), 'd_loss': d_loss.numpy()})
    print(f"Epoch {epoch+1} finished")


# 4. Interactive Generation
Run this cell to generate a specific digit of your choice!


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create buttons for digits 0-9
buttons = [widgets.Button(description=str(i), layout=widgets.Layout(width='45px')) for i in range(10)]
output = widgets.Output()

def on_button_click(button):
    digit = int(button.description)
    with output:
        output.clear_output(wait=True)
        noise = tf.random.normal([1, 100])
        label = tf.convert_to_tensor([[digit]], dtype=tf.int32) # shape (1,1)
        generated_img = generator([noise, label], training=False)
        plt.figure(figsize=(4, 4))
        plt.imshow(generated_img[0, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.title(f"GAN Generated: {digit}")
        plt.axis('off')
        plt.show()

for button in buttons:
    button.on_click(on_button_click)

# Display buttons in a grid layout
button_grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns='repeat(5, 45px)'))
display(widgets.VBox([widgets.Label('Select a digit to generate:'), button_grid, output]))